# California House Price Predictor — Colab Training

Run each cell top to bottom. At the end, download `xgb_california.pkl` and put it in your local `checkpoints/` folder.

In [ ]:
# 1. Install dependencies
!pip install xgboost wandb pyyaml scikit-learn matplotlib -q

In [ ]:
# 2. Upload housing.csv when prompted
from google.colab import files
uploaded = files.upload()  # select housing.csv from your computer

In [ ]:
# 3. Set up project structure
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('src', exist_ok=True)

# Move uploaded CSV
import shutil
if 'housing.csv' in uploaded:
    shutil.copy('housing.csv', 'data/raw/housing.csv')
    print('housing.csv moved to data/raw/')

In [ ]:
# 4. Write config (CPU mode for Colab)
config_yaml = '''
paths:
  raw_data: data/raw/housing.csv
  processed_data: data/processed/housing_processed.csv
  checkpoint_dir: checkpoints/
  checkpoint_name: xgb_california.pkl

features:
  numeric:
    - longitude
    - latitude
    - housing_median_age
    - total_rooms
    - total_bedrooms
    - population
    - households
    - median_income
    - rooms_per_household
    - bedrooms_per_room
    - population_per_household
    - dist_to_sf
    - dist_to_la
    - dist_to_san_diego
    - dist_to_sacramento
  categorical:
    - ocean_proximity
  target: median_house_value

preprocessing:
  test_size: 0.2
  random_state: 42
  fill_missing_strategy: median

model:
  name: xgboost
  params:
    n_estimators: 500
    max_depth: 6
    learning_rate: 0.05
    subsample: 0.8
    colsample_bytree: 0.8
    min_child_weight: 3
    gamma: 0.1
    reg_alpha: 0.1
    reg_lambda: 1.0
    random_state: 42
    eval_metric: rmse
    early_stopping_rounds: 50
    tree_method: hist

wandb:
  project: california-house-predictor
  entity: null
  log_feature_importance: true
  log_predictions: true

cities:
  san_francisco:
    lat: 37.7749
    lon: -122.4194
  los_angeles:
    lat: 34.0522
    lon: -118.2437
  san_diego:
    lat: 32.7157
    lon: -117.1611
  sacramento:
    lat: 38.5816
    lon: -121.4944
'''

with open('configs/config.yaml', 'w') as f:
    f.write(config_yaml)
print('Config written.')

In [ ]:
# 5. Write src/utils.py
utils_code = '''
import yaml, pickle, os
import numpy as np

def load_config(config_path="configs/config.yaml"):
    with open(config_path, "r") as f:
        return yaml.safe_load(f)

def save_checkpoint(model, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(model, f)
    print(f"Model saved to {path}")

def load_checkpoint(path):
    with open(path, "rb") as f:
        return pickle.load(f)

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))

def get_checkpoint_path(config):
    return os.path.join(config["paths"]["checkpoint_dir"], config["paths"]["checkpoint_name"])
'''
with open('src/utils.py', 'w') as f:
    f.write(utils_code)
open('src/__init__.py', 'w').close()
print('src/utils.py written.')

In [ ]:
# 6. WandB login
import wandb
wandb.login()  # Paste your API key when prompted

In [ ]:
# 7. Train the model
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import yaml, pickle, os

from src.utils import load_config, save_checkpoint, get_checkpoint_path, haversine_distance

config = load_config()
cities = config['cities']

# Load and clean
df = pd.read_csv(config['paths']['raw_data'])
df.fillna(df.median(numeric_only=True), inplace=True)

# Feature engineering
df['rooms_per_household'] = df['total_rooms'] / df['households']
df['bedrooms_per_room'] = df['total_bedrooms'] / df['total_rooms']
df['population_per_household'] = df['population'] / df['households']
df['dist_to_sf'] = haversine_distance(df['latitude'], df['longitude'], cities['san_francisco']['lat'], cities['san_francisco']['lon'])
df['dist_to_la'] = haversine_distance(df['latitude'], df['longitude'], cities['los_angeles']['lat'], cities['los_angeles']['lon'])
df['dist_to_san_diego'] = haversine_distance(df['latitude'], df['longitude'], cities['san_diego']['lat'], cities['san_diego']['lon'])
df['dist_to_sacramento'] = haversine_distance(df['latitude'], df['longitude'], cities['sacramento']['lat'], cities['sacramento']['lon'])

le = LabelEncoder()
df['ocean_proximity'] = le.fit_transform(df['ocean_proximity'].astype(str))

features = config['features']['numeric'] + config['features']['categorical']
target = config['features']['target']

X = df[features]
y = df[target]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

# Train
params = config['model']['params'].copy()
early_stop = params.pop('early_stopping_rounds', 50)

wandb.init(project=config['wandb']['project'], config=params)

model = xgb.XGBRegressor(**params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=early_stop, verbose=50)

y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)

print(f'\nRMSE: {rmse:,.0f}')
print(f'MAE:  {mae:,.0f}')
print(f'R²:   {r2:.4f}')

wandb.log({'RMSE': rmse, 'MAE': mae, 'R2': r2})
wandb.finish()

save_checkpoint(model, get_checkpoint_path(config))
print('Done!')

In [ ]:
# 8. Download the checkpoint
from google.colab import files
files.download('checkpoints/xgb_california.pkl')
print('Download started. Put this file in your local checkpoints/ folder.')